# SAE: RBO, Cosine Similarity, and Two-Concept Steering

This notebook is intentionally limited to three goals:

1. **RBO comparison** between the original SASRec ranking and the unsteered SAE model.
2. **Cosine similarity** between the original user representation $h$ and SAE reconstruction $\hat{h}$.
3. **Joint two-concept steering**: amplify one concept and suppress another, then report target exposure changes, non-target changes, and NDCG change.

The model/dataset paths and the top-1 concept-neuron discovery rule are kept from the original SAE notebook.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
import rbo

from recbole.quick_start import load_data_and_model

# Reproducible display
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


## 1. Load the base SASRec, SAE model, and concept metadata

In [ ]:
# Original SASRec model: used only as the reference for RBO.
base_config, base_model, _, _, _, _ = load_data_and_model(
    config_dict={
        'data_path': 'dataset/ml-1m',
        'dataset': 'ml-1m',
    },
    model_file='./saved/sasrec_ml-1m.pth',
)

# SAE model and the evaluation loaders used for the remaining analyses.
config, SAE_model, dataset, train_data, valid_data, test_data = load_data_and_model(
    config_dict={
        'data_path': 'dataset/ml-1m',
        'dataset': 'ml-1m',
        'base_path': './saved/sasrec_ml-1m.pth',
    },
    model_file='saved/SASRec_Mon_SAE-Aug-31-2026_11-05-06.pth',
)

base_model.eval()
SAE_model.eval()
device = config['device']

# Make sure any built-in one-neuron steering is disabled for baseline analyses.
if hasattr(SAE_model, 'steer_neuron_idx'):
    SAE_model.steer_neuron_idx = None
if hasattr(SAE_model, 'steer_add'):
    SAE_model.steer_add = 0.0
if hasattr(SAE_model, 'steer_scale'):
    SAE_model.steer_scale = 1.0

saved_concepts = pd.read_pickle(
    '/home/mvarasteh/post-hoc/dataset/ml-1m/saved_concept_individual_items.pkl'
)
concept_names = list(saved_concepts['concept_names'])
item_concepts_np = np.asarray(saved_concepts['item_concepts'])

print('Device:', device)
print('Number of concepts:', len(concept_names))
print('Item-concept matrix:', item_concepts_np.shape)


## 2. RBO comparison: SASRec vs unsteered SAE

In [ ]:
@torch.no_grad()
def get_top_k_recommendations(model, data_loader, k=20):
    """Return {user_id: top-k internal item ids}, with seen items masked."""
    model.eval()
    model_device = next(model.parameters()).device
    recs = {}

    for batch in data_loader:
        if isinstance(batch, (tuple, list)):
            interaction = batch[0]
            history_index = batch[1] if len(batch) > 1 else None
        else:
            interaction = batch
            history_index = None

        interaction = interaction.to(model_device)
        scores = model.full_sort_predict(interaction)
        batch_size = interaction[model.USER_ID].shape[0]
        scores = scores.view(batch_size, -1).clone()

        # Padding item.
        scores[:, 0] = -torch.inf

        # Prefer RecBole's full-sort history mask when it is supplied.
        if history_index is not None:
            history_index_device = tuple(
                x.to(model_device) if torch.is_tensor(x) else x
                for x in history_index
            )
            scores[history_index_device] = -torch.inf
        elif hasattr(model, 'ITEM_SEQ') and model.ITEM_SEQ in interaction:
            # Fallback for loaders that do not expose history_index.
            item_seq = interaction[model.ITEM_SEQ]
            for row in range(batch_size):
                seen = item_seq[row][item_seq[row] != 0]
                scores[row, seen] = -torch.inf

        topk = torch.topk(scores, k=k, dim=1).indices
        user_ids = interaction[model.USER_ID].detach().cpu().tolist()

        for user_id, items in zip(user_ids, topk):
            recs[int(user_id)] = items.detach().cpu()

    return recs


def compare_rbo(recs_a, recs_b, k=20, p=0.9):
    """Per-user Rank-Biased Overlap for two recommendation dictionaries."""
    common_users = sorted(set(recs_a) & set(recs_b))
    scores = []

    for user_id in common_users:
        a = recs_a[user_id][:k].tolist()
        b = recs_b[user_id][:k].tolist()

        # The installed rbo package used by Steering.ipynb supports RankingSimilarity.
        similarity = rbo.RankingSimilarity(a, b)
        try:
            value = similarity.rbo()
        except TypeError:
            # Compatibility with versions whose rbo() uses its default p internally.
            value = similarity.rbo()

        # Defensive handling for rbo implementations that return a dict.
        if isinstance(value, dict):
            value = value.get('ext', value.get('res', value.get('min')))

        scores.append(float(value))

    scores = np.asarray(scores, dtype=float)
    report = {
        'users': len(common_users),
        'mean_rbo': scores.mean(),
        'median_rbo': np.median(scores),
        'std_rbo': scores.std(),
    }

    print(f"Users compared: {report['users']}")
    print(f"Mean RBO@{k}:   {report['mean_rbo']:.4f}")
    print(f"Median RBO@{k}: {report['median_rbo']:.4f}")
    print(f"Std RBO@{k}:    {report['std_rbo']:.4f}")

    return scores, report


RBO_K = 20
RBO_P = 0.9

recs_sasrec = get_top_k_recommendations(base_model, test_data, k=RBO_K)
recs_sae = get_top_k_recommendations(SAE_model, test_data, k=RBO_K)

rbo_scores, rbo_report = compare_rbo(
    recs_sasrec,
    recs_sae,
    k=RBO_K,
    p=RBO_P,
)


## 3. Cosine similarity: $h$ vs. SAE reconstruction $\hat{h}$

In [ ]:
@torch.no_grad()
def collect_h_hhat_sae(model, data_loader, device=None):
    """Collect original SASRec user representation h and SAE reconstruction h_hat."""
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    H, H_hat = [], []

    for batch in data_loader:
        interaction = batch[0] if isinstance(batch, (tuple, list)) else batch
        interaction = interaction.to(device)

        item_seq = interaction[model.ITEM_SEQ]
        item_seq_len = interaction[model.ITEM_SEQ_LEN]

        h = model._encode(item_seq, item_seq_len)
        z = model.sae.encode(h)
        h_hat = model.sae.decode(z)

        H.append(h.detach().cpu())
        H_hat.append(h_hat.detach().cpu())

    return torch.cat(H, dim=0), torch.cat(H_hat, dim=0)


def cosine_similarity_report(H, H_hat):
    assert H.shape == H_hat.shape, f'Shape mismatch: {H.shape} vs {H_hat.shape}'

    cos = F.cosine_similarity(H.float(), H_hat.float(), dim=-1)
    report = {
        'cosine_mean': cos.mean().item(),
        'cosine_median': cos.median().item(),
        'cosine_std': cos.std().item(),
    }

    print(f'Samples: {H.shape[0]}, dimension: {H.shape[1]}')
    print(f"Cosine mean:   {report['cosine_mean']:.4f}")
    print(f"Cosine median: {report['cosine_median']:.4f}")
    print(f"Cosine std:    {report['cosine_std']:.4f}")
    return report


H, H_hat = collect_h_hhat_sae(SAE_model, test_data, device=device)
cosine_report = cosine_similarity_report(H, H_hat)


In [ ]:
item_concepts

## 4. Two-concept steering setup

In [ ]:
# Change only these values for a different paired intervention.
AMPLIFY_CONCEPT = 'Drama'
SUPPRESS_CONCEPT = 'Comedy'

# The original SAE notebook multiplies the amplified neuron and the suppressed
# neuron. The names below make that behavior explicit.
AMPLIFY_SCALE = 3.0
SUPPRESS_SCALE = 0.5

K = 10

# For ML-1M genre steering, the first 18 concepts are the genre group.
# Change this list if steering a different concept family.
CONCEPT_GROUP_INDICES = list(range(18))


@torch.no_grad()
def top1_neuron_for_concept(model, concept_name, concept_names, item_concepts):
    """Top-1 SAE neuron by mean activation over items belonging to the concept."""
    concept_idx = concept_names.index(concept_name)

    item_ids = np.where(item_concepts[:, concept_idx] > 0)[0]
    item_ids = item_ids[item_ids != 0]  # remove RecBole padding item
    if len(item_ids) == 0:
        raise ValueError(f'No items found for concept: {concept_name}')

    item_ids_t = torch.as_tensor(item_ids, dtype=torch.long, device=device)
    item_embeddings = model.item_embedding.weight[item_ids_t]
    activations = model.sae.encode(item_embeddings)
    mean_activation = activations.mean(dim=0)

    neuron_idx = int(torch.argmax(mean_activation).item())
    neuron_value = float(mean_activation[neuron_idx].item())
    return neuron_idx, neuron_value


AMPLIFY_NEURON, amplify_mean_activation = top1_neuron_for_concept(
    SAE_model, AMPLIFY_CONCEPT, concept_names, item_concepts_np
)
SUPPRESS_NEURON, suppress_mean_activation = top1_neuron_for_concept(
    SAE_model, SUPPRESS_CONCEPT, concept_names, item_concepts_np
)

print(f'Amplify  {AMPLIFY_CONCEPT}: neuron {AMPLIFY_NEURON} '
      f'(mean item activation={amplify_mean_activation:.4f})')
print(f'Suppress {SUPPRESS_CONCEPT}: neuron {SUPPRESS_NEURON} '
      f'(mean item activation={suppress_mean_activation:.4f})')

if AMPLIFY_NEURON == SUPPRESS_NEURON:
    raise ValueError(
        f'{AMPLIFY_CONCEPT} and {SUPPRESS_CONCEPT} map to the same SAE neuron '
        f'({AMPLIFY_NEURON}); they cannot be independently steered.'
    )


## 5. Joint steering and evaluation

In [ ]:
# Reconstruct the item side once; only the USER latent is steered.
with torch.no_grad():
    all_item_embeddings = SAE_model.item_embedding.weight.detach()
    all_item_latents = SAE_model.sae.encode(all_item_embeddings)
    reconstructed_item_embeddings = SAE_model.sae.decode(all_item_latents).clone()
    reconstructed_item_embeddings[0] = 0.0


@torch.no_grad()
def two_concept_full_sort_predict(
    model,
    interaction,
    amplify_neuron,
    amplify_scale,
    suppress_neuron,
    suppress_scale,
    reconstructed_item_embeddings,
):
    """Jointly amplify one SAE neuron and suppress another for the user vector."""
    item_seq = interaction[model.ITEM_SEQ]
    item_seq_len = interaction[model.ITEM_SEQ_LEN]

    h = model._encode(item_seq, item_seq_len)
    z_user = model.sae.encode(h).clone()

    z_user[:, int(amplify_neuron)] *= float(amplify_scale)
    z_user[:, int(suppress_neuron)] *= float(suppress_scale)
    z_user = torch.clamp(z_user, min=0.0)

    h_hat = model.sae.decode(z_user)
    return h_hat @ reconstructed_item_embeddings.t()


@torch.no_grad()
def compute_ndcg_from_topk(topk, positive_u, positive_i, k):
    """User-level NDCG@K for a RecBole full-sort batch."""
    device = topk.device
    batch_size = topk.size(0)

    positive_u = positive_u.to(device=device, dtype=torch.long)
    positive_i = positive_i.to(device=device, dtype=torch.long)

    relevance = torch.zeros((batch_size, k), dtype=torch.float32, device=device)
    if positive_i.numel() > 0:
        matches = topk[positive_u] == positive_i.unsqueeze(1)
        positive_pair_idx, rank_idx = torch.where(matches)
        if positive_pair_idx.numel() > 0:
            user_rows = positive_u[positive_pair_idx]
            relevance[user_rows, rank_idx] = 1.0

    ranks = torch.arange(1, k + 1, device=device, dtype=torch.float32)
    discounts = 1.0 / torch.log2(ranks + 1.0)
    dcg = (relevance * discounts).sum(dim=1)

    num_positives = torch.bincount(positive_u, minlength=batch_size)
    valid_mask = num_positives > 0
    ideal_len = torch.clamp(num_positives, max=k)

    cumulative_discounts = torch.cumsum(discounts, dim=0)
    idcg = torch.zeros(batch_size, dtype=torch.float32, device=device)
    idcg[valid_mask] = cumulative_discounts[ideal_len[valid_mask] - 1]

    ndcg = torch.zeros_like(dcg)
    ndcg[valid_mask] = dcg[valid_mask] / idcg[valid_mask]
    return ndcg, valid_mask


@torch.no_grad()
def evaluate_condition(
    model,
    test_data,
    item_concepts,
    concept_names,
    concept_indices,
    reconstructed_item_embeddings,
    k,
    amplify_scale=1.0,
    suppress_scale=1.0,
):
    """Evaluate one baseline/steered condition: exposure fractions + NDCG."""
    model.eval()
    n_items = test_data.dataset.item_num
    selected_names = [concept_names[i] for i in concept_indices]

    concept_tensor = torch.as_tensor(
        item_concepts[:, concept_indices],
        dtype=torch.float32,
        device=device,
    )

    concept_fraction_sum = torch.zeros(
        len(concept_indices), dtype=torch.float64, device=device
    )
    total_users = 0
    ndcg_sum = 0.0
    ndcg_users = 0

    for batch in tqdm(test_data, leave=False):
        
        interaction, history_index, positive_u, positive_i = batch
        interaction = interaction.to(device)

        scores = two_concept_full_sort_predict(
            model=model,
            interaction=interaction,
            amplify_neuron=AMPLIFY_NEURON,
            amplify_scale=amplify_scale,
            suppress_neuron=SUPPRESS_NEURON,
            suppress_scale=suppress_scale,
            reconstructed_item_embeddings=reconstructed_item_embeddings,
        ).view(-1, n_items)

        scores[:, 0] = -torch.inf
        if history_index is not None:
            history_index_device = tuple(
                x.to(device) if torch.is_tensor(x) else x
                for x in history_index
            )
            scores[history_index_device] = -torch.inf

        topk = torch.topk(scores, k=k, dim=1).indices

        labels = concept_tensor[topk]
        concept_fraction_sum += labels.mean(dim=1).sum(dim=0).double()
        total_users += topk.size(0)

        batch_ndcg, valid_mask = compute_ndcg_from_topk(
            topk, positive_u, positive_i, k
        )
        ndcg_sum += batch_ndcg[valid_mask].sum().item()
        ndcg_users += valid_mask.sum().item()

    mean_fractions = (concept_fraction_sum / total_users).cpu().numpy()
    return {
        'fractions': dict(zip(selected_names, mean_fractions)),
        'ndcg': ndcg_sum / ndcg_users,
    }


## 6. Run baseline vs. simultaneous two-concept steering

In [ ]:
baseline_result = evaluate_condition(
    model=SAE_model,
    test_data=test_data,
    item_concepts=item_concepts_np,
    concept_names=concept_names,
    concept_indices=CONCEPT_GROUP_INDICES,
    reconstructed_item_embeddings=reconstructed_item_embeddings,
    k=K,
    amplify_scale=1.0,
    suppress_scale=1.0,
)

steered_result = evaluate_condition(
    model=SAE_model,
    test_data=test_data,
    item_concepts=item_concepts_np,
    concept_names=concept_names,
    concept_indices=CONCEPT_GROUP_INDICES,
    reconstructed_item_embeddings=reconstructed_item_embeddings,
    k=K,
    amplify_scale=AMPLIFY_SCALE,
    suppress_scale=SUPPRESS_SCALE,
)

# Exposure changes in percentage points.
delta_pp = {
    name: 100.0 * (
        steered_result['fractions'][name] - baseline_result['fractions'][name]
    )
    for name in baseline_result['fractions']
}

target_delta = (
    delta_pp[AMPLIFY_CONCEPT],
    delta_pp[SUPPRESS_CONCEPT],
)

non_target_delta = {
    name: value
    for name, value in delta_pp.items()
    if name not in {AMPLIFY_CONCEPT, SUPPRESS_CONCEPT}
}



largest_positive_non_target = max(non_target_delta.values())
largest_negative_non_target = min(non_target_delta.values())

baseline_ndcg = baseline_result['ndcg']
steered_ndcg = steered_result['ndcg']
relative_ndcg_change_pct = 100.0 * (
    (steered_ndcg - baseline_ndcg) / (baseline_ndcg + 1e-12)
)

final_summary = pd.DataFrame([
    {
        'Amplify / Suppress': f'{AMPLIFY_CONCEPT} / {SUPPRESS_CONCEPT}',
        'Target Delta (pp)': f'({target_delta[0]:+.1f}, {target_delta[1]:+.1f})',
        'Non-target Delta (pp)': (
            f'({largest_positive_non_target:+.1f}, '
            f'{largest_negative_non_target:+.1f})'
        ),
        f'Baseline NDCG@{K}': baseline_ndcg,
        f'Steered NDCG@{K}': steered_ndcg,
        'Delta NDCG (%)': relative_ndcg_change_pct,
    }
])

display(final_summary.round(4))

print('\nAll concept exposure changes (pp):')
display(
    pd.Series(delta_pp, name='Delta exposure (pp)')
      .sort_values(ascending=False)
      .to_frame()
      .round(2)
)


## Trying differene values for scaling and supressing factors

In [ ]:
# ============================================================
# Grid search for two-concept steering
# Tune on VALIDATION data, then evaluate the best pair on TEST.
# ============================================================

# Concepts stay fixed for this search
AMPLIFY_CONCEPT = 'Comedy'
SUPPRESS_CONCEPT = 'Drama'

K = 10

# ML-1M genre concepts
CONCEPT_GROUP_INDICES = list(range(18))

# ------------------------------------------------------------
# Search space
# Amplification: > 1 increases the selected neuron
# Suppression:   < 1 decreases the selected neuron
# ------------------------------------------------------------
AMPLIFY_SCALES = np.arange(1, 10.25, 0.25)

SUPPRESS_SCALES = np.arange(1, 0, -0.05)


# ============================================================
# 1. Find neurons ONCE
# ============================================================

AMPLIFY_NEURON, amplify_mean_activation = top1_neuron_for_concept(
    SAE_model,
    AMPLIFY_CONCEPT,
    concept_names,
    item_concepts_np
)

SUPPRESS_NEURON, suppress_mean_activation = top1_neuron_for_concept(
    SAE_model,
    SUPPRESS_CONCEPT,
    concept_names,
    item_concepts_np
)

print(
    f'Amplify  {AMPLIFY_CONCEPT}: neuron {AMPLIFY_NEURON} '
    f'(mean activation={amplify_mean_activation:.4f})'
)

print(
    f'Suppress {SUPPRESS_CONCEPT}: neuron {SUPPRESS_NEURON} '
    f'(mean activation={suppress_mean_activation:.4f})'
)

if AMPLIFY_NEURON == SUPPRESS_NEURON:
    raise ValueError(
        f'{AMPLIFY_CONCEPT} and {SUPPRESS_CONCEPT} map to the same SAE '
        f'neuron ({AMPLIFY_NEURON}).'
    )


# ============================================================
# 2. Helper: convert one result into steering metrics
# ============================================================

def steering_metrics(baseline_result, steered_result):

    # Exposure change in percentage points
    delta_pp = {
        name: 100.0 * (
            steered_result['fractions'][name]
            - baseline_result['fractions'][name]
        )
        for name in baseline_result['fractions']
    }


    # ---------------------------------------
    # TARGETS
    # ---------------------------------------

    amp_delta = delta_pp[AMPLIFY_CONCEPT]

    # Suppressed target should be NEGATIVE.
    suppress_delta = delta_pp[SUPPRESS_CONCEPT]

    # Positive numbers mean desired movement
    amplify_effect = amp_delta
    suppress_effect = suppress_delta

    # Mean strength of the two intended changes
    mean_target_effect = (
        amplify_effect + suppress_effect
    ) / 2.0

 
    # Conservative score:
    # both concepts must steer well.
    balanced_target_effect = min(
        amplify_effect,
        suppress_effect
    )

    # ---------------------------------------
    # NON-TARGETS
    # ---------------------------------------

    non_target_delta = {
        name: value
        for name, value in delta_pp.items()
        if name not in {
            AMPLIFY_CONCEPT,
            SUPPRESS_CONCEPT
        }
    }

    largest_positive_non_target = max(
        non_target_delta.values()
    )

    largest_negative_non_target = min(
        non_target_delta.values()
    )

    # Worst unintended absolute exposure shift
    max_abs_non_target = max(
        abs(v) for v in non_target_delta.values()
    )

    # Also useful as an overall collateral-change statistic
    mean_abs_non_target = np.mean(
        [abs(v) for v in non_target_delta.values()]
    )

    # ---------------------------------------
    # NDCG
    # ---------------------------------------

    baseline_ndcg = baseline_result['ndcg']
    steered_ndcg = steered_result['ndcg']

    relative_ndcg_change_pct = 100.0 * (
        (steered_ndcg - baseline_ndcg)
        / (baseline_ndcg + 1e-12)
    )

    # We care about LOSS.
    # If NDCG improves, loss is zero.
    ndcg_loss = max(
        0.0,
        -relative_ndcg_change_pct
    )

    return {
        'amp_delta': amp_delta,
        'suppress_delta': suppress_delta,

        'amplify_effect': amplify_effect,
        'suppress_effect': suppress_effect,

        'mean_target_effect': mean_target_effect,
        'balanced_target_effect': balanced_target_effect,

        'largest_positive_non_target':
            largest_positive_non_target,

        'largest_negative_non_target':
            largest_negative_non_target,

        'max_abs_non_target':
            max_abs_non_target,

        'mean_abs_non_target':
            mean_abs_non_target,

        'baseline_ndcg': baseline_ndcg,
        'steered_ndcg': steered_ndcg,

        'delta_ndcg_pct':
            relative_ndcg_change_pct,

        'ndcg_loss':
            ndcg_loss,

        'delta_pp': delta_pp,
    }


# ============================================================
# 3. VALIDATION baseline -- calculate only ONCE
# ============================================================

print('\nComputing validation baseline...')

valid_baseline = evaluate_condition(
    model=SAE_model,
    test_data=valid_data,
    item_concepts=item_concepts_np,
    concept_names=concept_names,
    concept_indices=CONCEPT_GROUP_INDICES,
    reconstructed_item_embeddings=reconstructed_item_embeddings,
    k=K,
    amplify_scale=1.0,
    suppress_scale=1.0,
)

print(
    f'Validation baseline NDCG@{K}: '
    f'{valid_baseline["ndcg"]:.6f}'
)




# ============================================================
# 4. GRID SEARCH
# ============================================================

search_rows = []

total_runs = (
    len(AMPLIFY_SCALES)
    * len(SUPPRESS_SCALES)
)

print(
    f'\nSearching {total_runs} '
    f'amplify/suppress combinations...\n'
)


for amp_scale in AMPLIFY_SCALES:

    for sup_scale in SUPPRESS_SCALES:

        steered = evaluate_condition(
            model=SAE_model,
            test_data=valid_data,
            item_concepts=item_concepts_np,
            concept_names=concept_names,
            concept_indices=CONCEPT_GROUP_INDICES,
            reconstructed_item_embeddings=
                reconstructed_item_embeddings,
            k=K,
            amplify_scale=amp_scale,
            suppress_scale=sup_scale,
        )

        m = steering_metrics(
            valid_baseline,
            steered
        )

        search_rows.append({
            'Amplify Scale': amp_scale,
            'Suppress Scale': sup_scale,

            'Amplify delta (pp)':
                m['amp_delta'],

            'Suppress delta (pp)':
                m['suppress_delta'],

            'Mean Target Effect':
                m['mean_target_effect'],

            'Balanced Target Effect':
                m['balanced_target_effect'],

            'Non-target +':
                m['largest_positive_non_target'],

            'Non-target -':
                m['largest_negative_non_target'],

            'Max |Non-target|':
                m['max_abs_non_target'],

            'Mean |Non-target|':
                m['mean_abs_non_target'],

            f'NDCG@{K}':
                m['steered_ndcg'],

            'delta_NDCG_perc':
                m['delta_ndcg_pct'],

            'NDCG Loss_perc':
                m['ndcg_loss'],
        })


results_df = pd.DataFrame(search_rows)

In [ ]:
# 1. Keep only valid rows
valid = results_df[
    (results_df["NDCG Loss_perc"] <= 3) &
    (results_df["Amplify delta (pp)"] > 0) &
    (results_df["Suppress delta (pp)"] < 0)
].copy()

# 2. Overall steering strength
valid["Steering_Score"] = (
    valid["Amplify delta (pp)"]
    + valid["Suppress delta (pp)"].abs()
)

# 3. Find the best row
best_row = valid.loc[valid["Steering_Score"].idxmax()]

print(best_row)

In [ ]:
# ============================================================
# 5. Select the best configuration
# ============================================================

# Maximum allowed collateral exposure change (percentage points)
MAX_NON_TARGET_PP = 2.0

# Maximum allowed relative NDCG loss (%)
MAX_NDCG_LOSS_PCT = 3.0


# ------------------------------------------------------------
# Keep only configurations that:
#   1. successfully amplify the amplified target
#   2. successfully suppress the suppressed target
#   3. keep non-target changes <= threshold
#   4. keep NDCG loss <= threshold
# ------------------------------------------------------------

acceptable = results_df[
    (results_df['Amplify delta (pp)'] > 0)
    &
    (results_df['Suppress delta (pp)'] < 0)
    &
    (results_df['Max |Non-target|'] <= MAX_NON_TARGET_PP)
    &
    (results_df['NDCG Loss_perc'] <= MAX_NDCG_LOSS_PCT)
].copy()


# ============================================================
# Rank acceptable configurations
# ============================================================

if len(acceptable) > 0:

    # Priority:
    # 1. Maximize the weaker target steering effect
    # 2. Minimize maximum non-target change
    # 3. Minimize NDCG loss
    acceptable = acceptable.sort_values(
        by=[
            'Balanced Target Effect',
            'Max |Non-target|',
            'NDCG Loss_perc'
        ],
        ascending=[
            False,   # larger target effect is better
            True,    # smaller non-target change is better
            True     # smaller NDCG loss is better
        ]
    )

    best_row = acceptable.iloc[0]

    print(
        f'Found {len(acceptable)} configurations '
        f'satisfying all constraints.'
    )

    ranked_to_display = acceptable


# ============================================================
# If nothing satisfies the constraints
# ============================================================

else:

    print(
        'No configuration satisfied all constraints.\n'
        'Falling back to configurations where both target '
        'directions are correct.'
    )

    ranked = results_df[
        (results_df['Amplify delta (pp)'] > 0)
        &
        (results_df['Suppress delta (pp)'] < 0)
    ].copy()

    ranked = ranked.sort_values(
        by=[
            'Balanced Target Effect',
            'Max |Non-target|',
            'NDCG Loss_perc'
        ],
        ascending=[
            False,
            True,
            True
        ]
    )

    if len(ranked) == 0:
        raise ValueError(
            'No configuration produced both positive amplification '
            'and negative suppression.'
        )

    best_row = ranked.iloc[0]
    ranked_to_display = ranked


# ============================================================
# Display top configurations
# ============================================================

display(
    ranked_to_display[
        [
            'Amplify Scale',
            'Suppress Scale',
            'Amplify delta (pp)',
            'Suppress delta (pp)',
            'Balanced Target Effect',
            'Non-target +',
            'Non-target -',
            'Max |Non-target|',
            'Mean |Non-target|',
            f'NDCG@{K}',
            'delta_NDCG_perc',
            'NDCG Loss_perc'
        ]
    ]
    .head(20)
    .round(4)
)


# ============================================================
# Save best scaling factors
# ============================================================

BEST_AMPLIFY_SCALE = float(
    best_row['Amplify Scale']
)

BEST_SUPPRESS_SCALE = float(
    best_row['Suppress Scale']
)


# ============================================================
# Print best configuration
# ============================================================

print('\nBEST VALIDATION CONFIGURATION')
print('--------------------------------')

print(
    f'Amplify concept  : {AMPLIFY_CONCEPT}'
)

print(
    f'Suppress concept : {SUPPRESS_CONCEPT}'
)

print(
    f'Amplify neuron   : {AMPLIFY_NEURON}'
)

print(
    f'Suppress neuron  : {SUPPRESS_NEURON}'
)

print(
    f'Amplify scale    : {BEST_AMPLIFY_SCALE}'
)

print(
    f'Suppress scale   : {BEST_SUPPRESS_SCALE}'
)

print(
    f'Target delta     : '
    f'({best_row["Amplify delta (pp)"]:+.2f}, '
    f'{best_row["Suppress delta (pp)"]:+.2f})'
)

print(
    f'Non-target delta : '
    f'({best_row["Non-target +"]:+.2f}, '
    f'{best_row["Non-target -"]:+.2f})'
)

print(
    f'Max |non-target| : '
    f'{best_row["Max |Non-target|"]:.2f} pp'
)

print(
    f'NDCG@{K}         : '
    f'{best_row[f"NDCG@{K}"]:.6f}'
)

print(
    f'delta NDCG       : '
    f'{best_row["delta_NDCG_perc"]:+.2f}%'
)

print(
    f'NDCG loss        : '
    f'{best_row["NDCG Loss_perc"]:.2f}%'
)